In [1]:
!pip install langchain


In [2]:
from google.colab import userdata
api_key = userdata.get('GROQ_API_KEY')
api_key

'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'

In [3]:
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.8 MB/s eta 0:00:00


In [4]:
!pip install ipykernel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 16.1 MB/s eta 0:00:00


In [13]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=api_key)
model

ChatGroq(profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7fb347ffb020>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7fb347ffb620>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [7]:
!pip install langchain_core

In [19]:
from langchain_core.messages import HumanMessage,SystemMessage
messages=[
    SystemMessage(content="Translate the following from English to French"),  #instructs how to ai to behave
    HumanMessage(content="Hello How are you?")  #what sentence to convert
]

result=model.invoke(messages)

In [16]:
result

AIMessage(content='Bonjour, comment allez-vous ?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 47, 'total_tokens': 55, 'completion_time': 0.036566364, 'completion_tokens_details': None, 'prompt_time': 0.001732057, 'prompt_tokens_details': None, 'queue_time': 0.053920829, 'total_time': 0.038298421}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb353-eb35-7951-835f-0bc6fded7f03-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 8, 'total_tokens': 55})

In [20]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

'Bonjour, comment allez-vous ?'

In [21]:
## sing LCEL-chain the components
chain=model|parser
chain.invoke(messages)
## first message will go to model and give output in its big format then it will go to parser to give required output

'Bonjour, comment allez-vous ?'

In [28]:
## prompt template
from langchain_core.prompts import ChatPromptTemplate
generic_template= "Translate the following into {language}:"

prompt=ChatPromptTemplate.from_messages(
    [
        ("system",generic_template),
        ("user","{text}")
    ]

)
prompt

ChatPromptTemplate(input_variables=['language', 'text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='Translate the following into {language}:'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='{text}'), additional_kwargs={})])

In [29]:
result=prompt.invoke({"language":"French","text":"Hello"})


In [30]:
result.to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [31]:
chain=prompt|model|parser
chain.invoke({"language":"French","text":"Hello"})

'Bonjour'

In [32]:
!pip install fastapi

In [33]:
!pip install uvicorn

In [34]:
!pip install langserve

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.5 MB/s eta 0:00:00


In [35]:
!pip install sse_starlette

In [39]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [42]:
%%writefile serve.py
from fastapi import FastAPI
from langchain_core.prompts import ChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langserve import add_routes
import os

api_key = os.environ["GROQ_API_KEY"]
model=ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=api_key)



## prompt template
from langchain_core.prompts import ChatPromptTemplate
generic_template= "Translate the following into {language}:"

prompt=ChatPromptTemplate.from_messages(
    [
        ("system",generic_template),
        ("user","{text}")
    ]

)

parser=StrOutputParser()

##create chain
chain=prompt|model|parser

## App definition
app=FastAPI(title="Langchain Server",
            version="1.0",
            description="A simple API server using Langchain runnable interfaces ")

## adding chain routes
add_routes(
    app,
    chain,
    path="/chain"
)

if __name__=="__main__":
  import uvicorn
  uvicorn.run(app,host="127.0.0.1",port=8000)

Overwriting serve.py


In [43]:
import subprocess

subprocess.Popen(["python", "serve.py"])

<Popen: returncode: None args: ['python', 'serve.py']>

In [44]:
!python serve.py

INFO:     Started server process [23230]
INFO:     Waiting for application startup.

     __          ___      .__   __.   _______      _______. _______ .______     ____    ____  _______
    |  |        /   \     |  \ |  |  /  _____|    /       ||   ____||   _  \    \   \  /   / |   ____|
    |  |       /  ^  \    |   \|  | |  |  __     |   (----`|  |__   |  |_)  |    \   \/   /  |  |__
    |  |      /  /_\  \   |  . `  | |  | |_ |     \   \    |   __|  |      /      \      /   |   __|
    |  `----./  _____  \  |  |\   | |  |__| | .----)   |   |  |____ |  |\  \----.  \    /    |  |____
    |_______/__/     \__\ |__| \__|  \______| |_______/    |_______|| _| `._____|   \__/     |_______|
    
LANGSERVE: Playground for chain "/chain/" is live at:
LANGSERVE:  │
LANGSERVE:  └──> /chain/playground/
LANGSERVE:
LANGSERVE: See all available routes at /docs/
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address alread

In [45]:
from google.colab import output
output.serve_kernel_port_as_window(8000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [47]:
!pip install pyngrok

In [48]:
from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))

public_url = ngrok.connect(8000)
print(public_url)

NgrokTunnel: "https://brewing-treadmill-raking.ngrok-free.dev" -> "http://localhost:8000"
